# Discourse Lab — reference notebook

A guided tour of the toolbox, in build order (see `TODO.txt` and
`discourse-lab-dev.md` §6). Each section is runnable on its own once the
previous one has executed — copy a section into your own notebook as a
starting point.

Dynamics are numeric; language (the last section) is an offline pass over
already-decided numeric state and never runs inside the simulation loop.


## Running in Google Colab

Open this notebook in Colab (GitHub tab -> `lrodeck/Network-Simulation`,
branch `claude/todos-continuation-tupw4f` -> `notebooks/demo.ipynb`, or File ->
Upload notebook), then run the cell below first — it installs the package
straight from this repo (not on PyPI, and not yet merged to `main`, hence the
pinned branch) and is a no-op outside Colab. Everything after it is
unchanged: `workspace()` already resolves to `/content/dlab` under Colab
automatically.


In [1]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q "git+https://github.com/lrodeck/Network-Simulation.git@claude/todos-continuation-tupw4f"


In [2]:
import os
import warnings
import dataclasses

import numpy as np
import polars as pl

from discourse_lab.config import Config

# A scratch workspace so this notebook never touches a real ~/dlab.
os.environ["DLAB_HOME"] = os.path.join(os.getcwd(), "dlab_demo")

# The demo population/tick counts below are deliberately tiny for speed, which
# makes the cascade size/depth caps bind far more often than they would at a
# realistic scale — expected here, not a sign of a bug. Silence the warning
# for this notebook; leave it on when calibrating a real run.
warnings.filterwarnings("ignore", message="cascade:")

SEED = 0
rng = np.random.default_rng(SEED)


## 1. Config

Every run is a pure function of `(Config, seed)`. Nested, frozen, and
structurally hashed — the hash is what artifact caching and sweep
resumability key off of. This notebook uses a small population and a short
run so every cell finishes in seconds; swap in the defaults (`Config()`) for
anything you intend to actually analyze.


In [3]:
cfg = Config()
cfg = dataclasses.replace(
    cfg,
    population=dataclasses.replace(cfg.population, n_users=600, n_topics=4),
    # drift defaults to "full"; turned off here so sections 1-8 isolate the
    # dynamics this notebook is actually demonstrating. Turned back on in §9.
    dynamics=dataclasses.replace(cfg.dynamics, n_ticks=30, kernel="homophily", ranker="affinity", drift="none"),
)
print("config hash:", cfg.hash())
print("stance dims:", cfg.stance_dims())
cfg


config hash: f8c5b673ea46a0e9ca9ae6d0a7f9f082
stance dims: 1


Config(population=PopulationConfig(n_users=600, n_topics=4, stance_dims=-1, archetype_weights=(), archetype_offsets=(), correlation_pairs=(), activity_sigma=1.2, pareto_alpha=2.3, topic_logit_sigma=1.0), graph=GraphConfig(generator='latent_space', mean_degree=40.0, homophily_beta=0.35, prominence_gamma=0.6, reciprocity=0.2, fanout_cap=400, knn_k=150, long_tie_fraction=0.1, sbm_blocks=0, sbm_homophily=0.8), dynamics=DynamicsConfig(n_ticks=30, posts_per_tick_rate=0.02, ticks_per_day=24, fatigue_decay=0.9, attention_budget=30.0, tau_position=6.0, inject_k=0, ranker='affinity', kernel='homophily', kernel_theta=(), hawkes_mu0=0.004, hawkes_ratio=0.6, hawkes_beta=1.5, max_thread_age=15, trend_eta=0.3, post_lifetime=5, rho_s=0.9, rho_sigma=0.9, cascade_depth_decay=0.7, max_cascade_depth=4, max_cascade_size=1000, drift='none', drift_lr=0.02, drift_lr_social=0.01, drift_ramp_ticks=50, ou_k=(), noise_sigma=0.002, llm_adjudication=False, snapshot_every=1, exposure_sample_rate=0.01), scenario=Scen

## 2. Population

Archetype mixture over a correlated Gaussian latent, transformed to target
marginals by a Gaussian copula (spec §2.1). `cached_population` reuses a
prior draw for this exact population sub-config + seed.


In [4]:
from discourse_lab.population import cached_population

pop = cached_population(cfg, seed=SEED, rng=rng)
print(f"{len(pop.trait_names)} traits x {cfg.population.n_users} users")
print("archetypes:", sorted(set(pop.archetype_names)))

archetype_of_user = np.array(pop.archetype_names)[pop.archetype_labels]
pl.DataFrame(
    {
        "archetype": archetype_of_user,
        "activity": pop.X_used[:, pop.trait_names.index("activity")],
        "prominence": pop.X_used[:, pop.trait_names.index("prominence")],
        "contrarianism": pop.X_used[:, pop.trait_names.index("contrarianism")],
    }
).group_by("archetype").agg(pl.all().mean()).sort("archetype")


25 traits x 600 users
archetypes: ['firebrand', 'institution', 'lurker', 'newcomer', 'poster']


archetype,activity,prominence,contrarianism
str,f64,f64,f64
"""firebrand""",2.767024,1.494404,0.480374
"""institution""",2.960572,46.030835,0.273703
"""lurker""",0.678882,1.563494,0.272066
"""newcomer""",2.205378,1.642837,0.245579
"""poster""",4.718272,1.668911,0.254656


## 3. Graph

`latent_space` (the default) connects users by homophily in stance/topic
space plus a preferential-attachment term on prominence, calibrated by
bisection to hit `mean_degree`. Swappable for `sbm`, `configuration_model`,
or `barabasi_albert` via `cfg.graph.generator` — each is a useful null model
for isolating what homophily itself contributes.


In [5]:
from discourse_lab.network import cached_graph
from discourse_lab.network.measures import degree_sequence, global_clustering

graph = cached_graph(cfg, seed=SEED, pop=pop, rng=rng)
deg = degree_sequence(graph.csr)
print(f"mean degree: {deg.mean():.1f} (target {cfg.graph.mean_degree})")
print(f"clustering coefficient: {global_clustering(graph.csr):.3f}")


mean degree: 50.5 (target 40.0)


clustering coefficient: 0.233


## 4. Stance editor widget

Draw a population's stance distribution by hand; the sampler preview shows
exactly what a copula draw from that curve looks like. Autosaves to
`scenarios/<name>.json` in the same schema `ScenarioConfig.from_editor_json`
reads, so a drawn scenario plugs directly back into a `Config`.


In [6]:
from discourse_lab.widgets import StanceEditorWidget

stance_editor = StanceEditorWidget(name="demo")
stance_editor


## 5. Running the simulation

`run_iter(cfg, seed)` is the generator core — timing, generation, exposure,
reaction, cascades, perception, and the discourse-state update, one tick at a
time (drift is off in this `cfg`; see §9). `cached_run`/`run` collect it
into a persisted, parquet-backed run directory.


In [7]:
from discourse_lab.runner import cached_run, load_run

run_dir = cached_run(cfg, seed=SEED)
handle = load_run(cfg, seed=SEED)
metrics = handle.metrics()
print("run dir:", os.path.relpath(run_dir))
metrics.select(["t", "n_posts", "n_exposures", "attention_gini", "bubble_index", "r_eff"]).tail(5)


run dir: dlab_demo/runs/f8c5b673ea46a0e9ca9ae6d0a7f9f082/0


t,n_posts,n_exposures,attention_gini,bubble_index,r_eff
i64,f64,f64,f64,f64,f64
25,251.0,2367.0,0.702729,0.994099,17.197719
26,209.0,2381.0,0.701191,0.994746,16.745485
27,244.0,2403.0,0.693508,0.995376,18.19226
28,243.0,2371.0,0.689931,0.994389,16.490089
29,299.0,2300.0,0.68293,0.99545,18.25087


In [8]:
# spec §5.1: the simulation is calibrated when this lands in [0.8, 0.95]
# without being imposed.
print("mean attention Gini:", metrics["attention_gini"].drop_nulls().mean())
# spec §5.1's target ranges assume a realistically sized, calibrated run;
# this notebook's tiny demo population may or may not land inside them.


mean attention Gini: 0.6750232801719359


## 6. Run monitor widget (live)

Consumes `run_iter` directly and plots measures as ticks complete — the
salience/agreement pair, bubble index, attention Gini, R_eff — so a config's
fate is visible within twenty ticks rather than only at the end.


In [9]:
from discourse_lab.runner import run_iter
from discourse_lab.widgets import RunMonitorWidget

monitor = RunMonitorWidget()
for state in run_iter(cfg, seed=SEED + 1):  # a fresh seed; population/graph artifacts are reused
    monitor.push(state)
monitor


## 7. Post-run metrics

A separate module over a completed run: distributions, not point estimates.
Every reported effect should be a difference against a matched
`kernel="null"` run (spec §5.3) — see the sweep in §8.


In [10]:
from discourse_lab.metrics import attention_inequality, bimodality_coefficient, stylized_facts_report

gini, top1_share = attention_inequality(np.random.default_rng(2).pareto(2.3, size=2000))
print(f"example attention inequality on a synthetic Pareto tail: gini={gini:.2f}, top-1% share={top1_share:.2f}")

stance_col = pop.trait_names.index(next(n for n in pop.trait_names if n.startswith("stance_")))
print("population stance bimodality coefficient:", round(bimodality_coefficient(pop.X_used[:, stance_col]), 3))

stylized_facts_report(attention_gini=float(metrics["attention_gini"].drop_nulls().mean()))


example attention inequality on a synthetic Pareto tail: gini=0.62, top-1% share=0.15
population stance bimodality coefficient: 0.352


{'attention_gini': {'value': 0.6750232801719359,
  'target': (0.8, 0.95),
  'in_range': False}}

## 8. Experiment 1 — kernel/ranker sweep vs. null

Every effect is measured against a matched `kernel="null"` run (same
population, graph, activity). Kept tiny here (2 kernels x 1 ranker x 3
seeds); bump `seeds` to 10+ for anything you'd actually report.


In [11]:
from discourse_lab.experiments import build_experiment1, run_experiment1, summarize_experiment1

cells_exp1 = build_experiment1(cfg, kernels=("homophily", "bandwagon"), rankers=("affinity",))
rows = run_experiment1(cells_exp1, seeds=[0, 1, 2])
summarize_experiment1(rows)


kernel,ranker,attention_gini_effect_mean,bubble_index_effect_mean,r_eff_effect_mean,agreement_effect_mean,salience_effect_mean,attention_gini_effect_std,bubble_index_effect_std,r_eff_effect_std,agreement_effect_std,salience_effect_std
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""homophily""","""affinity""",-0.003429,0.000002,0.064344,0.001728,0.002345,0.000318,0.00018,0.103009,0.001329,0.001486
"""bandwagon""","""affinity""",0.045981,0.001356,7.492891,0.004471,0.003751,0.000898,0.000094,0.185155,0.001691,0.000834


## 9. Drift (optional)

Two free channels (reinforcement on expression traits, social influence on
stance) plus Ornstein-Uhlenbeck mean-reversion. Off by default
(`dynamics.drift`); gains ramp linearly from 0 over `drift_ramp_ticks` when
it's on, so switching it on mid-analysis doesn't jolt the population.


In [12]:
cfg_drift = dataclasses.replace(
    cfg,
    dynamics=dataclasses.replace(cfg.dynamics, drift="full", drift_ramp_ticks=10, n_ticks=20),
)
final_metrics = list(run_iter(cfg_drift, seed=SEED + 2))[-1].metrics
print("last-tick metrics with drift on:", {k: round(v, 3) for k, v in final_metrics.items()})


last-tick metrics with drift on: {'n_posts': 304.0, 'n_exposures': 2024.0, 'n_engagements': 1822.0, 'attention_gini': 0.654, 'salience': 0.952, 'agreement': -0.089, 'bubble_index': 0.993, 'r_eff': 16.062}


## 10. LLM realization (optional, offline pass)

Never inside the tick. Quantization and prompt-building need no network and
always run below; the actual call to
[Ollama Cloud](https://ollama.com) only fires if `OLLAMA_API_KEY` is set.

```bash
export OLLAMA_API_KEY=...   # https://ollama.com/settings/keys
```


In [13]:
from discourse_lab.dynamics import ExpressionMap, generate_posts
from discourse_lab.llm.voice_card import build_voice_card_messages, fit_bands, user_bands

K, D = cfg.population.n_topics, cfg.stance_dims()
expr = ExpressionMap.build(pop.trait_names, K)
authors = rng.choice(cfg.population.n_users, size=5, replace=True)
demo_posts = generate_posts(authors, pop, expr, np.zeros(K), np.zeros((K, D)), eta=0.3, rng=rng)

bands = fit_bands(pop)
example_bands = user_bands(pop, bands, int(authors[0]))
print("quantized traits (never raw floats in the prompt):", example_bands)

messages = build_voice_card_messages(pop.archetype_names[pop.archetype_labels[authors[0]]], example_bands)
print()
print(messages[1]["content"])


quantized traits (never raw floats in the prompt): {'openness': 'high', 'conscientiousness': 'low', 'extraversion': 'medium', 'agreeableness': 'medium', 'neuroticism': 'very high', 'plasticity': 'medium', 'conviction': 'medium', 'contrarianism': 'medium', 'credulity': 'very high'}

Archetype: institution

Trait profile:
- openness: high
- conscientiousness: low
- extraversion: medium
- agreeableness: medium
- neuroticism: very high
- plasticity: medium
- conviction: medium
- contrarianism: medium
- credulity: very high

Return exactly this format, nothing else:
PERSONA: <three sentences describing who this person is and how they post>
TICS: <tic one>; <tic two>; <tic three>
REGISTER: <one line on formality/vocabulary/punctuation habits>


In [14]:
if os.environ.get("OLLAMA_API_KEY"):
    from discourse_lab.llm import OllamaCloudClient, realize

    client = OllamaCloudClient(model="gpt-oss:120b-cloud")
    texts = realize(client, cfg, demo_posts, pop, post_ids=[int(demo_posts.id[0])])
    print(texts)
else:
    print("OLLAMA_API_KEY not set — skipping the live call. The prompt above is what would be sent.")


OLLAMA_API_KEY not set — skipping the live call. The prompt above is what would be sent.
